# MagicTelescope classification

Your astrophysicist friend is doing an experimental project with a certain telescope. You can assume you don't know anything about astrophysics (if you do, you may be able to improve this laboratory session! Send me an email). Your friend says that there is a simulation study that classified readings from a telescope into signal and background.

Because your friend does not know about machine learning, the task has fallen to you. The experiment they are trying to replicate has the following features:

1. fLength: continuous (major axis of ellipse)
2. fWidth: continuous (minor axis of ellipse)
3. fSize: continuous (10-log of sum of content of all pixels)
4. fConc: continuous (ratio of sum of two highest pixels over fSize)
5. fConc1: continuous (ratio of highest pixel over fSize)
6. fAsym: continuous (distance from highest pixel to center, projected onto major axis)
7. fM3Long: continuous (3rd root of third moment along major axis)
8. fM3Trans: continuous (3rd root of third moment along minor axis)
9. fAlpha: continuous (angle of major axis with vector to origin)
10. fDist: continuous (distance from origin to center of ellipse)

The class to predict is `g` or `h`, meaning a gamma (signal) or a hadron (background).

> **Note:** in the provided files every column name carries a trailing colon (`fLength:`, `fAlpha:`, ...), a leftover from the original dataset format. Keep it in mind when selecting columns, e.g. `X["fAlpha:"]`.

It looks like a straightforward classification problem, with only continuous variables. Easy, right?

In [12]:
import pyarrow  # we will be using parquet files, install pyarrow if it is not installed
import pandas as pd
import numpy as np

In [20]:
X = pd.read_parquet("datasets/labsession/magictelescope-data.parquet")
y = pd.read_parquet("datasets/labsession/magictelescope-labels.parquet")
X.head()

,fLength:,fWidth:,fSize:,fConc:,fConc1:,fAsym:,fM3Long:,fM3Trans:,fAlpha:,fDist:
0,30.882418,10.377231,2.525830,0.425839,0.273766,-0.799959,-13.417567,-4.017957,1.153199,130.882060
1,24.490004,13.055657,2.448719,0.470369,0.257590,21.384489,9.137348,-7.733678,0.456730,155.349742
2,24.172198,15.974486,2.578024,0.415380,0.246582,20.951367,-7.942711,4.400141,0.538452,132.981735
3,45.609628,28.494486,3.394777,0.185186,0.100495,NaN,46.335543,NaN,0.220561,108.337855
4,14.653023,15.082040,2.230790,0.762594,0.379777,22.031874,-2.581721,-8.148603,1.090214,253.923009


Let's perform a cross validation to see if `LogisticRegression` works properly.

We have done this in class so many times that I am ashamed to ask you to write it. Here it is, same as always:

> **Tip:** `y` is loaded as a one-column DataFrame. scikit-learn accepts it but will complain with a `DataConversionWarning` ("A column-vector y was passed when a 1d array was expected"). You can get rid of it with `y = y.iloc[:, 0]` (or `y.squeeze()`) whenever you want.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate

clf_linear = LogisticRegression()
cross_validate(clf_linear, X, y, cv=3)

ValueError: 
All the 3 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/python/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 860, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/python/lib/python3.13/site-packages/sklearn/base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py", line 1437, in fit
    X, y = validate_data(
           ~~~~~~~~~~~~~^
        self,
        ^^^^^
    ...<5 lines>...
        accept_large_sparse=solver not in ["liblinear", "sag", "saga"],
        ^^^^^^^^^

... wait, that is new! Why is this failing?

Mmm, it seems that your friend's telescope data has some missing values.

If you read the error message, scikit-learn is trying to help you by pointing you to the [`impute` module documentation](https://scikit-learn.org/stable/modules/impute.html). Create a `SimpleImputer` transformer to address the issue.

Show that the `SimpleImputer` is working. A simple `.fit_transform()` call is enough, followed by something like `np.isnan(...).sum()` on the result (note that `.fit_transform()` returns a NumPy array; wrap it in a `pd.DataFrame` if you prefer to use `.head()` or `.isna().sum()`).

In [21]:
from sklearn.impute import SimpleImputer

X_impute = SimpleImputer().fit_transform(X)
np.isnan(X_impute).sum()

np.int64(0)

In [28]:
X_impute = pd.DataFrame(X_impute)
X_impute.columns = X.columns
X_impute.head()

,fLength:,fWidth:,fSize:,fConc:,fConc1:,fAsym:,fM3Long:,fM3Trans:,fAlpha:,fDist:
0,30.882418,10.377231,2.525830,0.425839,0.273766,-0.799959,-13.417567,-4.017957,1.153199,130.882060
1,24.490004,13.055657,2.448719,0.470369,0.257590,21.384489,9.137348,-7.733678,0.456730,155.349742
2,24.172198,15.974486,2.578024,0.415380,0.246582,20.951367,-7.942711,4.400141,0.538452,132.981735
3,45.609628,28.494486,3.394777,0.185186,0.100495,-4.044904,46.335543,0.092443,0.220561,108.337855
4,14.653023,15.082040,2.230790,0.762594,0.379777,22.031874,-2.581721,-8.148603,1.090214,253.923009


Of course, the next step should be to create a pipeline and try again. But... your sixth sense is tingling. Are all the values fixed now?

We forgot to explore the DataFrame! Let's address that now. Call `.describe()` on the DataFrame. Look closely at the statistics and distribution... isn't the column `fAlpha:` suspicious? Look at its values, poke around a little. Maybe there are some values that should not be like that. Use another `SimpleImputer` to fix them (hint: the `missing_values` parameter can be useful!).

In [29]:
X_impute.describe()

,fLength:,fWidth:,fSize:,fConc:,fConc1:,fAsym:,fM3Long:,fM3Trans:,fAlpha:,fDist:
count,15216.000000,15216.000000,15216.000000,15216.000000,15216.000000,15216.000000,15216.000000,15216.000000,15216.000000,15216.000000
mean,53.335476,22.146097,2.824680,0.380580,0.214787,-4.044904,10.533832,0.092443,-953.729411,194.239770
std,41.239291,17.727897,0.458849,0.178989,0.107927,57.323769,49.621455,20.188557,2937.966339,72.809510
min,3.625139,-0.343458,1.942853,0.013566,0.001466,-458.501619,-332.713881,-206.022877,-9999.000000,0.606700
25%,24.997715,12.104094,2.491616,0.241232,0.131579,-18.263942,-11.478877,-10.500945,0.060861,146.624716
50%,39.463860,17.703773,2.772587,0.368860,0.205520,1.801993,13.560105,0.092443,0.243607,194.239770
75%,67.517321,24.237917,3.078394,0.493599,0.278879,23.070218,33.819034,10.538514,0.730064,237.643073
max,333.838368,256.630293,5.170606,0.892843,0.675668,573.626994,239.046012,179.849515,1.576762,495.271355


In [33]:
X_impute = SimpleImputer(missing_values=-9999.0).fit_transform(X_impute)
X_impute = pd.DataFrame(X_impute)
X_impute.columns = X.columns
X_impute.describe()

,fLength:,fWidth:,fSize:,fConc:,fConc1:,fAsym:,fM3Long:,fM3Trans:,fAlpha:,fDist:
count,15216.000000,15216.000000,15216.000000,15216.000000,15216.000000,15216.000000,15216.000000,15216.000000,15216.000000,15216.000000
mean,53.335476,22.146097,2.824680,0.380580,0.214787,-4.044904,10.533832,0.092443,0.479605,194.239770
std,41.239291,17.727897,0.458849,0.178989,0.107927,57.323769,49.621455,20.188557,0.432577,72.809510
min,3.625139,-0.343458,1.942853,0.013566,0.001466,-458.501619,-332.713881,-206.022877,-0.009380,0.606700
25%,24.997715,12.104094,2.491616,0.241232,0.131579,-18.263942,-11.478877,-10.500945,0.112620,146.624716
50%,39.463860,17.703773,2.772587,0.368860,0.205520,1.801993,13.560105,0.092443,0.379143,194.239770
75%,67.517321,24.237917,3.078394,0.493599,0.278879,23.070218,33.819034,10.538514,0.730064,237.643073
max,333.838368,256.630293,5.170606,0.892843,0.675668,573.626994,239.046012,179.849515,1.576762,495.271355


Create a pipeline with the required `SimpleImputer` instances and the `LogisticRegression`. Run `cross_validate` and see if we are getting somewhere now.

In [41]:
from sklearn.pipeline import make_pipeline

model = make_pipeline(SimpleImputer(), SimpleImputer(missing_values=-9999.0), LogisticRegression(max_iter=10))

In [ ]:
cv = cross_validate(model, X, y, cv=3)
cv

/opt/python/lib/python3.13/site-packages/sklearn/utils/validation.py:1368: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/python/lib/python3.13/site-packages/sklearn/utils/validation.py:1368: DataConversionWarning: A column-vector y was passed when a 1d array was exp

{'fit_time': array([0.10406303, 0.09321547, 0.09813428]), 'score_time': array([0.01340413, 0.0108633 , 0.01172328]), 'test_score': array([0.77523659, 0.79357256, 0.77484227])}


/opt/python/lib/python3.13/site-packages/sklearn/utils/validation.py:1368: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


You will probably get a `ConvergenceWarning` (if you have not, use `LogisticRegression(max_iter=10)` to see the warning).

Answer the following questions _very briefly_. Write your answers in the markdown cell right below. You can run some tests yourself in the code cells after it (your playground) to make sure that your answers are correct:
- Can `max_iter` affect the quality of a model?
- Will a converged model always perform better (have a higher score) than a non-converged one?
- Given two converged models, one with a `StandardScaler` transformer and one without, would their scores differ significantly?
- Can the presence of a `StandardScaler` transformer affect the number of iterations?

...

In [ ]:
...

At this point we could do some more model selection, hyperparameter tuning (e.g. if using an ensemble), and so on. At the end, we would finally `.fit()` the model on all our training data to obtain the final model (this includes the transformers and everything, of course).

Now is the time to test the performance of the model on a separate testing set:

In [44]:
X_test = pd.read_parquet("datasets/labsession/magictelescope-testdata.parquet")
y_test = pd.read_parquet("datasets/labsession/magictelescope-testlabels.parquet")

Fit your pipeline on the whole training set (`X`, `y`), keep it in a variable called `model`, and score it with the test set: `model.score(X_test, y_test)`.

In [48]:
model = make_pipeline(SimpleImputer(), SimpleImputer(missing_values=-9999.0), LogisticRegression())
model.fit(X, y)

/opt/python/lib/python3.13/site-packages/sklearn/utils/validation.py:1368: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/opt/python/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('simpleimputer-1', ...), ('simpleimputer-2', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](2,)","['g','h']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['fLength:','fWidth:','fSize:',...,'fM3Trans:','fAlpha:','fDist:']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'mean'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For 

In [49]:
model.score(X_test, y_test)

0.3856466876971609

Wait, that score is awful!

What have you done?

Ok, joking aside: the original performance (given by cross validation) was close to 80%, and now this test dataset gives a score below 50% (worse than always predicting the most frequent class!). Surely there is something wrong with the dataset.

Explore the `X_test` DataFrame and compare how the data is distributed in both datasets. If you had a lot of time, now would be the moment to do a side-by-side violin plot and look for meaningful deviations between them. But for this simple example, it should be enough to look firmly at the `.describe()` output of both. Which column is _very_ different between `X_test` and the training data? Why?

Fix it and score the model again. Explain which column is different and why in the markdown cell after the code cells.

In [54]:
X_impute.describe()

,fLength:,fWidth:,fSize:,fConc:,fConc1:,fAsym:,fM3Long:,fM3Trans:,fAlpha:,fDist:
count,15216.000000,15216.000000,15216.000000,15216.000000,15216.000000,15216.000000,15216.000000,15216.000000,15216.000000,15216.000000
mean,53.335476,22.146097,2.824680,0.380580,0.214787,-4.044904,10.533832,0.092443,0.479605,194.239770
std,41.239291,17.727897,0.458849,0.178989,0.107927,57.323769,49.621455,20.188557,0.432577,72.809510
min,3.625139,-0.343458,1.942853,0.013566,0.001466,-458.501619,-332.713881,-206.022877,-0.009380,0.606700
25%,24.997715,12.104094,2.491616,0.241232,0.131579,-18.263942,-11.478877,-10.500945,0.112620,146.624716
50%,39.463860,17.703773,2.772587,0.368860,0.205520,1.801993,13.560105,0.092443,0.379143,194.239770
75%,67.517321,24.237917,3.078394,0.493599,0.278879,23.070218,33.819034,10.538514,0.730064,237.643073
max,333.838368,256.630293,5.170606,0.892843,0.675668,573.626994,239.046012,179.849515,1.576762,495.271355


In [45]:
X_test.describe()

,fLength:,fWidth:,fSize:,fConc:,fConc1:,fAsym:,fM3Long:,fM3Trans:,fAlpha:,fDist:
count,3804.000000,3804.000000,3804.000000,3804.000000,3804.000000,3804.000000,3804.000000,3804.000000,3804.000000,3804.000000
mean,53.185334,22.245245,2.822177,0.379409,0.213755,-4.857429,9.754499,0.762035,28.074333,192.096335
std,42.651649,18.944012,0.472743,0.179441,0.108124,59.623418,51.234827,21.231947,26.127724,74.520707
min,8.230400,0.000000,1.946800,0.013300,0.000300,-381.734000,-293.176200,-142.589400,0.000000,19.888000
25%,24.060575,11.930775,2.478600,0.240750,0.131150,-20.231300,-12.960700,-10.591400,5.559450,140.473250
50%,37.116600,17.047350,2.736700,0.353750,0.195450,4.155300,15.232200,0.952450,18.641300,188.472500
75%,69.446275,24.240725,3.084100,0.503525,0.283000,23.516200,34.876150,11.085100,46.863300,239.513100
max,290.927400,190.543200,5.323300,0.877800,0.637000,473.065400,231.446000,170.692000,90.000000,450.349000


`fAlpha:` is very different. For the train dataset, the values were between 0 and 1.58, while for the test dataset, they are between 0 and 90. Since this covariate represents an angle, I assume the discrepancy comes from the unit of the measurement.

## Extra content

_Because your professor is awful at estimating the length of laboratory sessions._

Unless you reach this point very quickly, consider these exercises optional.

We were able to fix the mismatch in the `fAlpha:` feature because its semantics were clear to us.

Imagine that we did not know the conversion factor. A plausible workaround is to do the standardization outside the pipeline, separately for each dataset (it can make sense if there is a lot of data, but it is not an ideal scenario).

Without using pipelines: apply the imputers and a `StandardScaler` to the training DataFrame and train the model on the result. Then standardize the (original, unfixed) test dataset with a **different** `StandardScaler`, fitted on the test data itself, and score the model on it.

The score should be similar to the one you got after fixing the units. This "trick" only works when both datasets cover almost all of the data space; this is desirable and somewhat expected for the training dataset, but there is no such guarantee at the inference stage.

In [ ]:
...

Go back to the point where we introduced the basic `SimpleImputer`. It might be enough in some scenarios (an alternative is to drop rows entirely). There are other imputer transformers, such as the `KNNImputer`. This nearest-neighbors-based imputer fills each missing value according to the nearest neighbors (computed using the other features). It is **much** slower (it must search the neighbors of every row with a `nan`), but it can bring benefits (often depending on whether there is a hidden correlation across the features, which may or may not be the case). Replace the `SimpleImputer` with a `KNNImputer` and compare the cross validation scores.

In [ ]:
...